In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from replay.metrics import Recall, Precision, HitRate
import faiss
from functools import reduce
import datasets
from tqdm import tqdm
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
dataset = load_from_disk(f"{DATA_PATH}/user_clicks_20230501")
polars_ds = dataset.to_polars()

In [3]:
index = faiss.read_index("data/item_index_tiny.faiss")

with open("data/item_ids", "rb") as fp:
    item_ids = pickle.load(fp)

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 641.07it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

all_clicks = (
    train_interactions
    .select(
        pl.col("user_id"),
        pl.col("c2_name"),
        pl.col("name"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
)

In [6]:
query_clicks_20 = (
    all_clicks
    .filter(pl.col("rn") <= 20)
    .select("name", "item_id")
    .unique()
)

In [7]:
names = query_clicks_20["name"].to_list()
ids = query_clicks_20["item_id"].to_list()

In [35]:
batch_size = 4096
threshold = 1.0
similar_items = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(names), batch_size)):
    cur_names = names[i:i+batch_size]
    cur_ids = ids[i:i+batch_size]
    queries = model.encode(cur_names, batch_size=batch_size, normalize_embeddings=True)
    dists, idx = index.search(queries, k=1000)
    selected_idx = np.where(dists < threshold, idx, -1)
    recs = replace_func(selected_idx)
    cur_similar_items = {cur_ids[i]: recs[i, :][(recs[i, :] != cur_ids[i]) & recs[i, :] != -1].tolist()[:40] for i in range(len(cur_ids))}
    similar_items = {**similar_items, **cur_similar_items}

100%|██████████| 392/392 [34:59<00:00,  5.36s/it]


In [36]:
#with open("data/similar_items_lst20_top40_threshold=1.0", "wb") as fp:   #Pickling
#    pickle.dump(similar_items, fp)

In [6]:
#with open("data/similar_items_lst5_top50", "rb") as fp:   # Unpickling
#    similar_items = pickle.load(fp)

In [37]:
def get_similar_items(row):
    return list(reduce(lambda x, y: x + y, [similar_items[item_id] for item_id in row["last_clicks"] if item_id in similar_items]))

In [38]:
user_last_clicks_20 = (
     all_clicks
    .filter(pl.col("rn") <= 20)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [39]:
recs_20 = (
    user_last_clicks_20
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [40]:
recs_stats = (
    recs_20
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
    .select(
        pl.mean("recs_count").alias("mean_recs_count"),
        pl.min("recs_count").alias("min_recs_count"),
        pl.max("recs_count").alias("max_recs_count")
    )
)

recs_stats

mean_recs_count,min_recs_count,max_recs_count
f64,i64,i64
406.642137,40,960


In [41]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs_20,
        on="user_id",
        how="inner"
    )
)

In [42]:
TOP_K_VALUES = [10, 100, 960]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def intersection(row):
    return len(set(row["recs"]) & set(row["future_clicks"]))

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [43]:
metrics # threshold = 1.0 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.01545,0.048158,0.135513,0.01193,0.004957,0.002078,0.093205,0.255854,0.535999,10724.0,29438.0,61671.0


In [34]:
metrics # threshold = 0.99 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.01534,0.047874,0.13481,0.011851,0.004926,0.002067,0.092527,0.254437,0.534409,10646.0,29275.0,61488.0


In [25]:
metrics # threshold = 0.98 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.015068,0.04711,0.132909,0.011641,0.004853,0.002042,0.09078,0.250778,0.530333,10445.0,28854.0,61019.0


In [16]:
metrics # threshold = 0.97 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.014529,0.045845,0.129708,0.011243,0.004737,0.001997,0.087417,0.244824,0.522945,10058.0,28169.0,60169.0


In [40]:
metrics # threshold = 0.96 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.01382,0.044137,0.125214,0.010688,0.00458,0.001938,0.082871,0.236889,0.512907,9535.0,27256.0,59014.0


In [31]:
metrics # threshold = 0.95 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.012932,0.042038,0.120106,0.010046,0.004407,0.001872,0.077413,0.227833,0.500747,8907.0,26214.0,57615.0


In [21]:
metrics # threshold = 0.9 40 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.00826,0.032221,0.094968,0.006886,0.003574,0.001556,0.052339,0.184733,0.436658,6022.0,21255.0,50241.0


In [22]:
metrics # threshold = 0.9, 20 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.008259,0.035865,0.073474,0.006889,0.003986,0.002677,0.052365,0.209138,0.382277,6025.0,24063.0,43984.0


In [39]:
metrics # threshold = 0.8, 20 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.003656,0.020966,0.044491,0.00341,0.002459,0.001699,0.02676,0.137009,0.274774,3079.0,15764.0,31615.0


In [41]:
metrics # не фильтруем по трешхолду, 20 ближайших

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007292,0.035024,0.067852,0.0071,0.004236,0.002637,0.052174,0.215291,0.371847,6003.0,24771.0,42784.0
